# Lab 3: Know when you don't know

**AI in Space Applications II · after Lesson 20 · about 40 minutes · guided**

**Goal.** Build the first piece of VEGA-1's trust layer: a score that says how unlike its training data a tile is, a line set on ordinary tiles, a test on stress tiles, and a safe-fail rule.

Requirement R5: an out-of-distribution score flags the stress tiles at the detection and false-alarm rates **you** state, and flagged tiles are never discarded: they are sent, and any beyond your quota are held on board for a person to decide.

**Stress tiles.** Real snow from the independent test set (another labelling team, 20 m pixels), and three kinds made here by damaging ordinary tiles: stripes from a failed detector line, saturation, and sun glint. The made ones are labelled "simulated" wherever they appear.

**You need** `lab1_model.pt` from Lab 1 (it is in your Drive folder if you ran Lab 1 in Colab with Drive connected). The data files `vega1_train.npz` and `vega1_test.npz` download by themselves.

**In Colab.** Opened this from the course's *Open in Colab* link? First choose **File → Save a copy in Drive**, or your edits are lost when you close the tab. When the setup cell asks to connect Google Drive, allow it: the notebook keeps your models and results in the Drive folder `AI_in_Space_II_labs`, where later labs and sessions find them. The data files download by themselves.

In [ ]:
import json
import os
import platform
import random
import shutil
import time

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as Fn

FAST = os.environ.get("VEGA1_FAST") == "1"        # set only by the course's own tests: a short run that checks the code
SEED = 0

KEEP_DIR = None                                   # your folder in Google Drive, when this runs in Colab
try:
    from google.colab import drive                # Colab only: Colab forgets files when a session ends, Drive does not
    drive.mount("/content/drive")
    KEEP_DIR = "/content/drive/MyDrive/AI_in_Space_II_labs"
    os.makedirs(KEEP_DIR, exist_ok=True)
    for name in os.listdir(KEEP_DIR):             # bring back what you saved in earlier labs and sessions
        if not os.path.exists(name) and os.path.isfile(os.path.join(KEEP_DIR, name)):
            shutil.copy2(os.path.join(KEEP_DIR, name), name)
    print("Your Drive folder:", KEEP_DIR, "| files found there:", ", ".join(sorted(os.listdir(KEEP_DIR))) or "none yet")
except ImportError:
    pass                                          # not in Colab: files stay beside the notebook
except Exception as e:
    print("Google Drive is not connected, so files last only for this session:", e)


def keep(*names):
    """Copy these files to your Drive folder, so the next session and the next lab find them."""
    if KEEP_DIR:
        for name in names:
            if os.path.exists(name):
                shutil.copy2(name, os.path.join(KEEP_DIR, name))
        print("Kept in your Drive folder:", ", ".join(names))


def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


set_seed()
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Python", platform.python_version(), "| PyTorch", torch.__version__, "| NumPy", np.__version__, "| device:", DEVICE)

In [ ]:
DATA_DIR = os.environ.get("VEGA1_DATA_DIR", ".")
DATA_URL = "https://github.com/Astronify-SpaceTech/ai-in-space-ii-labs/releases/download/data-v1/"   # the course's data files


def get_file(name):
    """Find a course data file: beside the notebook, else download it from the course's link, else upload it in Colab."""
    path = os.path.join(DATA_DIR, name)
    if os.path.exists(path):
        return path
    if DATA_URL:
        import urllib.request
        try:
            print(f"downloading {name} ...")
            urllib.request.urlretrieve(DATA_URL + name, path + ".part")
            os.replace(path + ".part", path)
            return path
        except Exception as e:
            print(f"download failed ({e}); choose the file you downloaded from the lesson page instead")
    try:
        from google.colab import files       # Colab only: choose the file you downloaded from the lesson page
        files.upload()
        return name
    except ImportError:
        raise FileNotFoundError(f"Put {name} beside this notebook (it is attached to the lab on the course page).")


d = np.load(get_file("vega1_train.npz"))
X = d["X"].astype(np.float32) * float(d["scale"]) + float(d["offset"])     # stored value x scale + offset = reflectance
y = d["y"].astype(np.int64)                      # 0 clear, 1 thick cloud, 2 thin cloud, 3 cloud shadow
region, patch = d["region"], d["patch"]
CLASSES = ["clear", "thick cloud", "thin cloud", "cloud shadow"]

clear_share = (y == 0).mean(axis=(1, 2))         # share of each chip that is usable ground
usable = clear_share >= 0.5                      # R2 counts chips with at least half their area usable
print(f"{len(X)} chips, {X.shape[1]} bands, {X.shape[2]} x {X.shape[3]} pixels, {len(np.unique(region))} regions, "
      f"{len(np.unique(patch))} patches")

In [ ]:
class Block(nn.Sequential):
    def __init__(self, cin, cout):
        super().__init__(nn.Conv2d(cin, cout, 3, padding=1), nn.BatchNorm2d(cout), nn.ReLU(inplace=True),
                         nn.Conv2d(cout, cout, 3, padding=1), nn.BatchNorm2d(cout), nn.ReLU(inplace=True))


class SmallUNet(nn.Module):
    """Input: (batch, 7, H, W) reflectance. Output: (batch, 4, H, W) class scores. H and W must divide by 4."""

    def __init__(self, bands=7, classes=4, widths=(16, 32, 64)):
        super().__init__()
        a, b, c = widths
        self.register_buffer("mean", torch.zeros(1, bands, 1, 1))
        self.register_buffer("std", torch.ones(1, bands, 1, 1))
        self.enc1, self.enc2, self.mid = Block(bands, a), Block(a, b), Block(b, c)
        self.dec2, self.dec1 = Block(c + b, b), Block(b + a, a)
        self.head = nn.Conv2d(a, classes, 1)

    def features(self, x):
        x = (x - self.mean) / self.std
        e1 = self.enc1(x)
        e2 = self.enc2(Fn.max_pool2d(e1, 2))
        m = self.mid(Fn.max_pool2d(e2, 2))
        return e1, e2, m

    def forward(self, x):
        e1, e2, m = self.features(x)
        d2 = self.dec2(torch.cat([Fn.interpolate(m, scale_factor=2.0, mode="nearest"), e2], dim=1))
        d1 = self.dec1(torch.cat([Fn.interpolate(d2, scale_factor=2.0, mode="nearest"), e1], dim=1))
        return self.head(d1)


def train_model(X_train, y_train, make_model=SmallUNet, epochs=None, seed=SEED):
    """Train a model on the given chips and return it. About a minute on a GPU, a few minutes on a CPU."""
    set_seed(seed)
    epochs = epochs or (1 if FAST else 6)
    model = make_model().to(DEVICE)
    model.mean.copy_(torch.tensor(X_train.mean(axis=(0, 2, 3))).view(1, -1, 1, 1))      # normalise with the training
    model.std.copy_(torch.tensor(X_train.std(axis=(0, 2, 3)) + 1e-6).view(1, -1, 1, 1))  # chips' own statistics
    opt = torch.optim.Adam(model.parameters(), lr=2e-3)
    xt, yt = torch.from_numpy(X_train), torch.from_numpy(y_train)
    t0 = time.time()
    for epoch in range(epochs):
        model.train()
        order = torch.randperm(len(xt))
        total = 0.0
        for k in range(0, len(order), 16):
            idx = order[k:k + 16]
            xb, yb = xt[idx].to(DEVICE), yt[idx].to(DEVICE)
            if random.random() < 0.5:                     # a mirror flip: the same chip, seen another way
                xb, yb = xb.flip(-1), yb.flip(-1)
            logits = model(xb)
            step = yb.shape[-1] // logits.shape[-1]       # a model with a coarser output is scored on coarser labels
            loss = Fn.cross_entropy(logits, yb[:, ::step, ::step])
            opt.zero_grad()
            loss.backward()
            opt.step()
            total += float(loss) * len(idx)
        print(f"  epoch {epoch + 1}/{epochs}: loss {total / len(order):.3f} ({time.time() - t0:.0f} s)")
    return model.eval()


@torch.no_grad()
def predict(model, X_any, batch=32):
    """Class probabilities for every pixel, at the chip's own size: (chips, 4, H, W)."""
    out = []
    for k in range(0, len(X_any), batch):
        xb = torch.from_numpy(X_any[k:k + batch]).to(DEVICE)
        p = torch.softmax(model(xb), dim=1)
        if p.shape[-1] != xb.shape[-1]:
            p = Fn.interpolate(p, size=xb.shape[-2:], mode="nearest")
        out.append(p.cpu().numpy())
    return np.concatenate(out)

In [ ]:
def scores(model, X_test, y_test):
    """Cloud IoU (thick and thin cloud together), usable-area error in points, and pixel accuracy."""
    prob = predict(model, X_test)
    pred = prob.argmax(axis=1)
    pred_cloud, true_cloud = np.isin(pred, (1, 2)), np.isin(y_test, (1, 2))
    iou = (pred_cloud & true_cloud).sum() / max((pred_cloud | true_cloud).sum(), 1)
    area_error = np.abs(prob[:, 0].mean(axis=(1, 2)) - (y_test == 0).mean(axis=(1, 2))).mean() * 100
    return {"cloud_iou": float(iou), "usable_area_error_points": float(area_error),
            "pixel_accuracy": float((pred == y_test).mean())}


def region_split(region, share=0.25, seed=SEED):
    """Hold out whole regions: (train indices, test indices, the held-out regions)."""
    regions = np.random.default_rng(seed).permutation(np.unique(region))
    test_regions = regions[:max(1, int(len(regions) * share))]
    is_test = np.isin(region, test_regions)
    return np.where(~is_test)[0], np.where(is_test)[0], test_regions

## 1. State your rates first, and commit them

Before you test anything, write down the three numbers R5 asks for. Lesson 20's worked example used a 5% false-alarm budget and a quota of 10%: a fair start. Commit `PASS_MARKS.md` before you run section 4.

In [ ]:
DETECT_TARGET, FALSE_ALARM, QUOTA = ____, ____, ____     # share of stress tiles to flag; share of ordinary tiles flagged; daily quota
with open("PASS_MARKS.md", "a", encoding="utf-8") as f:
    f.write(f"- R5: flag at least {DETECT_TARGET:.0%} of each kind of stress tile at a false-alarm rate of {FALSE_ALARM:.0%}; "
            f"send flagged tiles up to {QUOTA:.0%} of a day's tiles, hold the rest. Written {time.strftime('%Y-%m-%d %H:%M:%S %Z')}.\n")
print("R5 commitments saved to PASS_MARKS.md")

<details><summary><b>Stuck?</b> A hint, not the answer</summary>

These are commitments, made before you test. Each is a share between 0 and 1: how many stress tiles you promise to flag, how many ordinary tiles you'll accept flagging by mistake, and how many flagged tiles a day you can afford to send. Lesson 20's worked example is a fair start. Write your three numbers, and why, in `PASS_MARKS.md`, and commit them.

</details>

## 2. Load the model, and the stress tiles

In [ ]:
ck = torch.load("lab1_model.pt", map_location="cpu")
model = SmallUNet(widths=tuple(ck["widths"]))
model.load_state_dict(ck["state"])
model.eval()
test_regions = np.array(ck["test_regions"])
is_test = np.isin(region, test_regions)
train_idx, test_idx = np.where(~is_test)[0], np.where(is_test)[0]

t = np.load(get_file("vega1_test.npz"))
XT = t["X"].astype(np.float32) * float(t["scale"]) + float(t["offset"])
snow_flag = t["snow"].astype(bool)
print(f"independent test set: {len(XT)} chips, {snow_flag.sum()} from subscenes tagged snow")

rng = np.random.default_rng(SEED)
base = X[test_idx]                                  # ordinary held-out tiles, to damage


def stripes(x):
    out = x.copy()
    out[:, :, ::6] = out.max()                      # every sixth detector column stuck bright (simulated)
    return out


def saturation(x):
    return np.minimum(x * 3.0, 1.0)                 # a gain error: everything bright clips at the top (simulated)


def glint(x):
    out = x.copy()
    h, w = x.shape[-2:]
    yy, xx = np.mgrid[:h, :w]
    spot = np.exp(-(((yy - h / 2) ** 2 + (xx - w / 3) ** 2) / (2 * (h / 5) ** 2)))
    return out + 0.8 * spot[None, None]             # a bright, soft-edged reflection of the Sun (simulated)


stress = {"snow (real)": XT[snow_flag], "stripes (simulated)": stripes(base),
          "saturation (simulated)": saturation(base), "glint (simulated)": glint(base)}
stress = {k: v.astype(np.float32) for k, v in stress.items()}
for k, v in stress.items():
    print(f"{k}: {len(v)} tiles")

## 3. Two candidate scores

- **Softmax confidence** (the tempting one): one minus the model's average confidence in its winning class.
- **Distance from home**: the model's deepest features, averaged over the tile, compared with the training tiles' features. A large Mahalanobis distance means "unlike anything I trained on" (Lesson 20).

In [ ]:
@torch.no_grad()
def deep_features(x_any, batch=32):
    out = []
    for k in range(0, len(x_any), batch):
        _, _, m = model.features(torch.from_numpy(x_any[k:k + batch]))
        out.append(torch.cat([m.mean(dim=(2, 3)), m.std(dim=(2, 3))], dim=1).numpy())
    return np.concatenate(out)


def softmax_score(x_any):
    return 1.0 - predict(model, x_any).max(axis=1).mean(axis=(1, 2))


home = deep_features(X[train_idx])
mu = home.mean(axis=0)
cov = np.cov(home, rowvar=False) + 1e-3 * np.eye(home.shape[1])
cov_inv = np.linalg.inv(cov)


def distance_score(x_any):
    d = deep_features(x_any) - mu
    return np.sqrt(np.einsum("ij,jk,ik->i", d, ____, d))       # the Mahalanobis distance: which matrix goes in the middle?

<details><summary><b>Stuck?</b> A hint, not the answer</summary>

The Mahalanobis distance weighs each direction by how much the training features spread along it, so a step along a narrow direction counts for more (Lesson 20). That needs the covariance turned upside down: which variable computed above holds it?

</details>

## 4. Set the line on ordinary tiles, then test

The line is set so that a share `FALSE_ALARM` of ordinary held-out tiles is flagged, and no more. Then every kind of stress tile is scored on its own.

**Checkpoint.** Softmax confidence lets most stress tiles through: the network is sure of itself on tiles it has never seen. The distance score should catch more of at least some kinds. Do not be surprised if **snow** gets past both: without shortwave infrared, the model sees snow as cloud, so its features look familiar. That is the sensor gap, found by your own test. Report each kind separately, and give snow its own line in the risk register and its own safe-fail rule: an average hides exactly what a review board needs to see.

In [ ]:
ordinary = X[test_idx]
results = {}
for name, score in (("softmax", softmax_score), ("distance", distance_score)):
    s_home = score(ordinary)
    line = float(np.quantile(s_home, ____))        # the line that flags a share FALSE_ALARM of ordinary tiles
    results[name] = {"line": line, "false_alarm": float((s_home > line).mean()),
                     "detected": {k: float((score(v) > line).mean()) for k, v in stress.items()}}
    print(f"{name}: false alarms {results[name]['false_alarm']:.0%} | " +
          " | ".join(f"{k} {v:.0%}" for k, v in results[name]["detected"].items()))
best = results["distance"]
meets = {k: v >= DETECT_TARGET for k, v in best["detected"].items()}
print("\nR5 with the distance score:", {k: ("meets" if ok else "MISSES") for k, ok in meets.items()})

<details><summary><b>Stuck?</b> A hint, not the answer</summary>

`np.quantile(x, q)` returns the value below which a share q of `x` lies. You want ordinary tiles to cross the line only a share FALSE_ALARM of the time. So what share must lie *below* it?

</details>

## 5. The safe-fail rule

Write it in two sentences an operator could follow at three in the morning. The reference rule is below; yours may differ, but a flagged tile is **never** discarded.

In [ ]:
SAFE_FAIL = ("A tile the monitor flags is sent, whatever the cloud model says, until flagged tiles reach the day's quota. "
             "Flagged tiles beyond the quota are held on board, and a person on the ground decides at the next pass.")


def decide(cloud_says_send, flagged, sent_flagged_today, quota_tiles):
    if not flagged:
        return "send" if cloud_says_send else "discard"
    return "send" if sent_flagged_today < quota_tiles else "hold for a person"


assert decide(False, True, 0, 10) == "send" and decide(False, True, 10, 10) == "hold for a person"
print(SAFE_FAIL)

## 6. Your turn: an alarm budget

VEGA-1 takes about about 12 pictures a day of 72 tiles each. Suppose operators can properly check 20 alarms in a day (Lesson 19). How many false alarms does your line send them on an ordinary day, and does it fit? If not, what would you change: the line, the quota, or the way alarms are grouped?

In [ ]:
tiles_per_day = 12 * 72
false_alarms_per_day = ____ * ____          # the false-alarm rate times the tiles screened in a day
print(f"about {false_alarms_per_day:.0f} false alarms a day against a budget of 20 -> "
      f"{'fits' if false_alarms_per_day <= 20 else 'does NOT fit'}")

<details><summary><b>Stuck?</b> A hint, not the answer</summary>

A rate times a count. The rate is the share of ordinary tiles your chosen score flags (it's in the best result above); the count is how many tiles are screened in a day (defined on the line above).

</details>

## 7. Check, then break, and save

**Check:** on ordinary tiles, the share flagged is the rate you set. **Break:** move the line to the median of the ordinary tiles, and watch half of all ordinary tiles become alarms.

In [ ]:
s_home = distance_score(ordinary)
print(f"flagged at your line: {(s_home > best['line']).mean():.0%} (set {FALSE_ALARM:.0%}); "
      f"at the median: {(s_home > np.median(s_home)).mean():.0%}")
with open("lab3_results.json", "w", encoding="utf-8") as f:
    json.dump({"commitments": {"detect": DETECT_TARGET, "false_alarm": FALSE_ALARM, "quota": QUOTA}, "scores": results,
               "meets": meets, "false_alarms_per_day": false_alarms_per_day, "safe_fail": SAFE_FAIL}, f, indent=1)
print("saved lab3_results.json")
keep("lab3_results.json")

## Stretch, the guardian desk (optional, not graded)

- **Orbits.** Fetch one satellite's orbit data from CelesTrak as OMM, propagate it with `sgp4` for five days, and compare with a newer record: how far off was the old one?
- **Collision probability.** Compute P꜀ for a close approach in ESA's Kelvins challenge data (CC BY 4.0), then scale its uncertainty from a tenth to ten times and plot the dilution curve.
- **Telemetry.** Draw a control chart on one OPS-SAT channel and count event-wise precision and recall, not point-wise.

## Hand in

1. Make sure every cell has run, top to bottom, with its outputs showing.
2. **File → Download → Download .ipynb**, and upload that file on the lab's page, with `lab3_results.json` from your Drive folder `AI_in_Space_II_labs`.
3. In the notes box, say which AI tools you used, if any.

Submitting marks the lab complete, so you can go on to the next lesson while it's graded. How it's graded is in the lab's instructions.